In [ ]:
# Notebook overview: 
# This notebook builds module of class ShotGradientEstimator used in simulator_final. 
# This module uses quantum machine learning (QML) parameter-shift rule to compute simulator gradient estimation. 
# This implementation avoids Barren plateaus. 

import numpy as np
class ShotGradientEstimator:
    def __init__(self, circuit, noise_model, H, shots, lambda0):
        self.circuit = circuit
        self.noise_model = noise_model
        self.H = H
        self.shots = shots
        self.lambda0 = lambda0

    def estimate_cost(self, theta):
        _, lam, dW = self.noise_model.sample_path(self.lambda0)
        w = self.noise_model.rn_weight(lam, dW)
        vals = []
        for _ in range(self.shots):
            self.circuit.run(theta)
            vals.append(self.circuit.expectation(self.H))
        return w * np.mean(vals)

    def estimate_gradient(self, theta, idx, shift):
        theta_plus = theta.copy()
        theta_minus = theta.copy()
        theta_plus[idx] += shift
        theta_minus[idx] -= shift
        c_plus = self.estimate_cost(theta_plus)
        c_minus = self.estimate_cost(theta_minus)
        return 0.5 * (c_plus - c_minus)
    
# For example usage, see gradients_api.ipynb